# Colab Local LLM Classifier
Runs the classifier stage locally on Colab through a vLLM OpenAI-compatible server and writes classifier-only prediction parquet files to Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/data/llm-gatekeeping'
REPO_URL = 'https://github.com/noamdwc/llm-gatekeeping.git'
REPO_DIR = '/content/llm-gatekeeping'
BRANCH = 'zero_shot_nlp_attack'

SPLIT = 'val'
LIMIT = 5  # Set to None for the full split.
MODEL_ID = 'meta/llama-3.1-8b-instruct'
TENSOR_PARALLEL_SIZE = 1
GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = 4096
BATCH_SIZE = 1
CHECKPOINT_EVERY = 50
OUTPUT_SUFFIX = 'colab_local_classifier'

HF_CACHE_DIR = f'{DRIVE_ROOT}/cache/huggingface'
SPLITS_DIR = f'{DRIVE_ROOT}/data/processed/splits'
PREDICTIONS_DIR = f'{DRIVE_ROOT}/data/processed/predictions'
CHECKPOINT_PATH = f'{PREDICTIONS_DIR}/llm_checkpoint_{SPLIT}_{OUTPUT_SUFFIX}.parquet'
OUTPUT_PATH = f'{PREDICTIONS_DIR}/llm_predictions_{SPLIT}_{OUTPUT_SUFFIX}.parquet'
VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'

import os

os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_HUB_CACHE'] = f'{HF_CACHE_DIR}/hub'
os.environ['TRANSFORMERS_CACHE'] = f'{HF_CACHE_DIR}/transformers'
os.environ['HF_DATASETS_CACHE'] = f'{HF_CACHE_DIR}/datasets'
print(f'Hugging Face cache: {HF_CACHE_DIR}')
print(f'Output path: {OUTPUT_PATH}')

In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Using existing repo at {REPO_DIR}')

os.chdir(REPO_DIR)
print('Repo:', os.getcwd())

subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q "vllm>=0.6.0" "openai>=1.0.0"

In [ ]:
from pathlib import Path
import subprocess
import time

import requests
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Colab GPU is unavailable. Enable Runtime > Change runtime type > GPU.')

Path(PREDICTIONS_DIR).mkdir(parents=True, exist_ok=True)
Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)

server_cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_ID,
    '--host', '127.0.0.1',
    '--port', '8000',
    '--tensor-parallel-size', str(TENSOR_PARALLEL_SIZE),
    '--gpu-memory-utilization', str(GPU_MEMORY_UTILIZATION),
    '--max-model-len', str(MAX_MODEL_LEN),
]

server_cmd_preview = 'python -m vllm.entrypoints.openai.api_server'
print('Starting vLLM:', ' '.join(server_cmd))
vllm_proc = subprocess.Popen(server_cmd)

deadline = time.time() + 600
last_error = None
while time.time() < deadline:
    try:
        response = requests.get(f'{VLLM_BASE_URL}/models', timeout=5)
        if response.ok:
            print('vLLM server is ready')
            break
        last_error = f'HTTP {response.status_code}: {response.text[:200]}'
    except Exception as exc:
        last_error = repr(exc)
    if vllm_proc.poll() is not None:
        raise RuntimeError(f'vLLM exited early with code {vllm_proc.returncode}')
    time.sleep(5)
else:
    raise RuntimeError(f'vLLM server did not become ready: {last_error}')

In [ ]:
import requests

models_response = requests.get(f'{VLLM_BASE_URL}/models', timeout=10)
models_response.raise_for_status()
print(models_response.json())

In [ ]:
from pathlib import Path

import pandas as pd

from src.llm_classifier.constants import NLP_TYPES
from src.llm_classifier.llm_classifier import HierarchicalLLMClassifier
from src.llm_classifier.llm_classifier import build_few_shot_examples
from src.llm_classifier.prompts import build_classifier_messages
from src.utils import build_sample_id, load_config

cfg = load_config()
text_col = cfg['dataset']['text_col']
label_col = cfg['dataset']['label_col']

train_path = Path(SPLITS_DIR) / 'train.parquet'
eval_path = Path(SPLITS_DIR) / f'{SPLIT}.parquet'
if not train_path.exists():
    raise FileNotFoundError(f'Missing train split: {train_path}')
if not eval_path.exists():
    raise FileNotFoundError(f'Missing eval split: {eval_path}')

train_df = pd.read_parquet(train_path)
eval_df = pd.read_parquet(eval_path)

for path, df, required in [
    (train_path, train_df, {text_col, label_col}),
    (eval_path, eval_df, {text_col}),
]:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{path} missing required columns: {sorted(missing)}')

if LIMIT is not None:
    eval_df = eval_df.head(int(LIMIT)).copy()
else:
    eval_df = eval_df.copy()

eval_df['sample_id'] = eval_df[text_col].apply(build_sample_id)
few_shot, used_ids = build_few_shot_examples(train_df, cfg)
print(f'Train rows: {len(train_df):,}; eval rows: {len(eval_df):,}; few-shot pairs: {len(few_shot)}')

In [ ]:
import json

from openai import OpenAI

client = OpenAI(base_url=VLLM_BASE_URL, api_key='EMPTY')
VALID_NLP_TYPES = set(NLP_TYPES) | {'none'}


def build_static_few_shot_messages(text: str) -> list[dict]:
    messages = []
    for benign_text, attack_text, attack_type in few_shot:
        messages.append({'role': 'user', 'content': f'INPUT_PROMPT:\n{benign_text}'})
        messages.append({
            'role': 'assistant',
            'content': json.dumps({
                'label': 'benign',
                'confidence': 95,
                'nlp_attack_type': 'none',
                'evidence': '',
                'reason': 'No active attempt to override instructions, exfiltrate data, or hijack tools.',
            }),
        })
        if attack_type in NLP_TYPES:
            evidence = ''
            adv_reason = f'Perturbed tokens characteristic of {attack_type} adversarial attack.'
        else:
            evidence = attack_text[:80]
            adv_reason = f'Contains {attack_type} obfuscation; active adversarial prompt detected.'
        messages.append({'role': 'user', 'content': f'INPUT_PROMPT:\n{attack_text}'})
        messages.append({
            'role': 'assistant',
            'content': json.dumps({
                'label': 'adversarial',
                'confidence': 84,
                'nlp_attack_type': attack_type if attack_type in NLP_TYPES else 'none',
                'evidence': evidence,
                'reason': adv_reason,
            }),
        })
    return messages


def extract_completion_logprobs(response) -> list[dict] | None:
    return HierarchicalLLMClassifier._extract_completion_logprobs(response.model_dump())


def normalize_classifier_payload(payload: dict, raw_response_text: str | None, token_logprobs: list[dict] | None) -> dict:
    label = payload.get('label', '') if payload else ''
    if label not in ('benign', 'adversarial', 'uncertain'):
        label = 'adversarial'

    nlp_attack_type = payload.get('nlp_attack_type', 'none') if payload else 'none'
    if nlp_attack_type not in VALID_NLP_TYPES:
        nlp_attack_type = 'none'
    if label != 'adversarial':
        nlp_attack_type = 'none'

    evidence = payload.get('evidence', '') if payload else ''
    if label != 'adversarial':
        evidence = ''

    confidence = HierarchicalLLMClassifier._normalize_confidence(
        payload.get('confidence', 50) if payload else 50
    )
    category = HierarchicalLLMClassifier._derive_category(label, nlp_attack_type)

    return {
        'label': label,
        'confidence': confidence,
        'nlp_attack_type': nlp_attack_type,
        'evidence': evidence,
        'reason': payload.get('reason', '') if payload else '',
        '_token_logprobs': token_logprobs,
        '_provider_name': 'vllm-local',
        '_model_name': MODEL_ID,
        '_raw_response_text': raw_response_text,
        '_parse_success': bool(payload),
        'clf_label': label,
        'clf_category': category,
        'clf_confidence': confidence,
        'clf_evidence': evidence,
        'clf_nlp_attack_type': nlp_attack_type,
        'clf_provider_name': 'vllm-local',
        'clf_model_name': MODEL_ID,
        'clf_raw_response_text': raw_response_text,
        'clf_parse_success': bool(payload),
        'clf_token_logprobs': token_logprobs,
    }


def parse_classifier_json(raw_response_text: str | None) -> dict:
    if raw_response_text is None:
        return {}
    try:
        payload = json.loads(raw_response_text)
        if isinstance(payload, str):
            payload = json.loads(payload)
        return payload if isinstance(payload, dict) else {}
    except json.JSONDecodeError:
        return {}


def classify_text(text: str) -> dict:
    messages = build_classifier_messages(text, build_static_few_shot_messages(text))
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=cfg['llm']['temperature'],
        max_tokens=cfg['llm']['max_tokens_classifier'],
        response_format={'type': 'json_object'},
        logprobs=True,
        top_logprobs=cfg['llm'].get('top_logprobs', 5),
    )
    raw_response_text = response.choices[0].message.content
    token_logprobs = extract_completion_logprobs(response)
    payload = parse_classifier_json(raw_response_text)
    return normalize_classifier_payload(payload, raw_response_text, token_logprobs)